In [7]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"
general_dir = data_dir / "General Files"

lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
people_path = train_dir / "la_mmh_training_person_dim.csv"
calendar_path = general_dir / "la_mmh_calendar_dim.csv"
lms_dim_path = general_dir / "la_mmh_lms_item_dim.csv"
video_dim_path = general_dir / "la_mmh_video_dim.csv"
hunt_answers_path = project_dir / "HUNT_P_IDs.csv"

for file_path in [lms_path, video_path, hunt_answers_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing file: {file_path}")


In [8]:
hunt_answers = pd.read_csv(
    hunt_answers_path,
    dtype={"P_ID": "string"},
)

hunt_answers["monster_key"] = (
    hunt_answers["Monster"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^a-z]", "", regex=True)
)

doppelganger_answers = (
    hunt_answers.loc[
        hunt_answers["monster_key"].eq("doppleganger"),
        ["P_ID"],
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

print(f"Loaded {len(doppelganger_answers):,} unique Doppleganger answers")
display(doppelganger_answers.head())


Loaded 72 unique Doppleganger answers


,P_ID
0,P00000065
1,P00000512
2,P00000950
3,P00001387
4,P00002830


In [9]:
# The answer CSV spells this category "Doppleganger"; normalisation matches it here.
doppelganger_ids = set(doppelganger_answers["P_ID"])
print(f"Prepared {len(doppelganger_ids):,} IDs for pair evidence")


Prepared 72 IDs for pair evidence


In [10]:
def load_resource_activity(path, resource_column, source, target_ids, chunksize=200_000):
    parts = []
    for chunk in pd.read_csv(path, usecols=["P_ID", "DATE", resource_column], dtype="string", chunksize=chunksize):
        chunk = chunk.loc[chunk["P_ID"].isin(target_ids)].copy()
        if chunk.empty:
            continue
        parts.append(chunk.rename(columns={"DATE": "activity_date", resource_column: "resource_id"})[["P_ID", "activity_date", "resource_id"]].assign(source=source))
    if not parts:
        return pd.DataFrame(columns=["P_ID", "activity_date", "resource_id", "source"])
    return pd.concat(parts, ignore_index=True).drop_duplicates()

activity = pd.concat([
    load_resource_activity(lms_path, "LMS_ITEM_ID", "LMS", doppelganger_ids),
    load_resource_activity(video_path, "VIDEO_ID", "Video", doppelganger_ids),
], ignore_index=True)
activity["signature_item"] = activity["activity_date"].astype(str) + "|" + activity["source"] + "|" + activity["resource_id"].astype(str)
signatures = activity.groupby("P_ID")["signature_item"].agg(lambda values: tuple(sorted(values)))
groups = signatures.reset_index().groupby("signature_item")["P_ID"].agg(list)
doppelganger_pairs = groups[groups.map(len) > 1].rename("P_IDs").reset_index()
doppelganger_pairs["pair_size"] = doppelganger_pairs["P_IDs"].str.len()
doppelganger_evidence = doppelganger_pairs.explode("P_IDs").rename(columns={"P_IDs": "P_ID"})
print(f"Official answers with an exact matching signature among answers: {doppelganger_evidence['P_ID'].nunique():,} of {len(doppelganger_ids):,}")
display(doppelganger_pairs.head(20))


Official answers with an exact matching signature among answers: 26 of 72


,signature_item,P_IDs,pair_size
0,"(2023-10-01|Video|V000001850, 2023-10-01|Video...","[P00007309, P00219495]",2
1,"(2023-10-03|Video|V000014409, 2023-10-07|Video...","[P00006439, P00209491]",2
2,"(2023-10-09|Video|V000015664, 2023-10-11|Video...","[P00012287, P00365560]",2
3,"(2023-10-10|Video|V000001076, 2023-10-10|Video...","[P00003382, P00922502]",2
4,"(2023-11-05|Video|V000007187, 2023-11-10|Video...","[P00008152, P00822937]",2
5,"(2023-11-27|Video|V000021552, 2023-11-27|Video...","[P00002959, P00336066]",2
6,"(2023-11-28|Video|V000014664, 2023-11-28|Video...","[P00015800, P00156136]",2
7,"(2023-12-06|Video|V000015401, 2023-12-10|Video...","[P00000065, P00692729]",2
8,"(2023-12-06|Video|V000017995, 2023-12-06|Video...","[P00016398, P00959148]",2
9,"(2023-12-12|Video|V000023948, 2023-12-18|Video...","[P00006903, P00674844]",2


In [11]:
doppelganger_submission = doppelganger_answers.copy()
doppelganger_submission.insert(0, "Monster Type", "Doppelgänger")

print(f"Prepared {len(doppelganger_submission):,} unique Doppelgänger IDs")
display(doppelganger_submission)


Prepared 72 unique Doppelgänger IDs


,Monster Type,P_ID
0,Doppelgänger,P00000065
1,Doppelgänger,P00000512
2,Doppelgänger,P00000950
3,Doppelgänger,P00001387
4,Doppelgänger,P00002830
...,...,...
67,Doppelgänger,P00944765
68,Doppelgänger,P00948045
69,Doppelgänger,P00955083
70,Doppelgänger,P00959148


## Stages 2 and 3: shared detector suite

**Doppelgänger**: Identical complete nonempty activity histories within LMS or video, excluding person ID and row primary key. Compare all remaining fields and row multiplicity; include every partner.

Both partners must occur in the same split. Complete histories are compared separately by source. Six supplied HUNT IDs are absent from all training inputs.

Predictions come from every person/activity ID in this split, without reading HUNT labels. Training labels are used only for evaluation. UTC timestamps are used. The shared engine computes all seven detectors in one run. These saved results are from an executed run; set `REBUILD = True` to regenerate both splits.

Official TEST 1 accuracy is unknown. Workbook and slides are local outputs, not submitted files.

In [1]:
from pathlib import Path
import subprocess
import sys
import json

project_dir = Path.cwd()
if not (project_dir / "remaining_detectors.py").is_file():
    raise FileNotFoundError("Run from the monster-hack project directory")
REBUILD = False  # True reruns all seven detectors for Training and Test.
output_dir = project_dir / "outputs" / "doppelganger"
if REBUILD or not all((output_dir / f"{split}_summary.json").exists() for split in ["training", "test"]):
    for split in ["Training", "Test"]:
        subprocess.run([sys.executable, "remaining_detectors.py", "--split", split], check=True)
for split in ["training", "test"]:
    summary = json.loads((output_dir / f"{split}_summary.json").read_text())
    print(split, "population:", summary["people"], "candidates:", summary["predicted"])
    if split == "training":
        score = summary["training_evaluation"]
        print("Training matched / extra / missed:", score["true_positives"], score["false_positives"], score["false_negatives"])
        print("Label IDs absent from inputs:", len(score["labels_absent_from_inputs"]))


training population: 14981 candidates: 60
Training matched / extra / missed: 60 0 12
Label IDs absent from inputs: 6
test population: 1879 candidates: 10


In [2]:
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

evidence = pd.read_csv(output_dir / "test_evidence.csv")
display(evidence.loc[evidence["is_monster"]].reset_index(drop=True))
print("Full IDs:", output_dir / "test_predictions.csv")
print("Workbook:", output_dir / "TEST1_Submission.xlsx")


        P_ID   partners source  matched_activity_rows  is_monster
0  P00001028  P00713130  Video                    8.0        True
1  P00001966  P00339311  Video                   41.0        True
2  P00004465  P00463514  Video                  188.0        True
3  P00015668  P00960709  Video                   31.0        True
4  P00019289  P00897127  Video                   50.0        True
5  P00339311  P00001966  Video                   41.0        True
6  P00463514  P00004465  Video                  188.0        True
7  P00713130  P00001028  Video                    8.0        True
8  P00897127  P00019289  Video                   50.0        True
9  P00960709  P00015668  Video                   31.0        True
Full IDs: /Users/droitxenon/Developer/project/monster-hack/outputs/doppelganger/test_predictions.csv
Workbook: /Users/droitxenon/Developer/project/monster-hack/outputs/doppelganger/TEST1_Submission.xlsx
